# E8 step 1: build the VAR-PEM horizon graphs, without LiNGAM (Kaggle, CPU)

Builds the graphs of **E8** (`S_h = P ⊙ |Ψ_h|`) and of its ablation **E8V** (`S_h = |Ψ_h|`) for h = 1, 3, 6, 12
(5, 15, 30, 60 minutes), top-25 sources per sensor. It uses only the VAR coefficients and PEM's probability matrix P.
**No DirectLiNGAM runs and no B0 file is opened**: the build stops if a LiNGAM module is imported or a file that looks
like a LiNGAM result is opened, and the check is printed and saved. Nothing is trained here; E8 / E8V training is step 2,
in `ablation-study.ipynb`.

**Before running** (Add Input)
1. The **PEM inputs dataset** (`pem-inputs`): `coefficients.npy`, `sensors.npy`, `metadata.json` and the **uniform-prior PEM
   result of the E4 / E5 setting** (`pem_result.npz`). The same dataset may also hold `lingam_result.npz`; it is not read.
2. Optional: the **artifacts dataset**, only to report how much the E8 graphs overlap the stored E5 graphs (comparison only).
3. Settings: **Accelerator = None (CPU)**, **Internet = On** (the code is cloned from GitHub). The GBA dataset is not needed.
4. **Save Version -> Save & Run All (Commit)**. Expected time: about 10-20 minutes.

Download `e8_output/` from the *Output* tab, upload the `E8__h*.npz` and `E8V__h*.npz` files to the artifacts dataset,
then train in `ablation-study.ipynb` with `EXPERIMENT = "E8"` (or `"E8V"`) and `HORIZON` 1, 3, 6 or 12.

## Settings

In [ ]:
import os

REPO_URL    = "https://github.com/prof-sanjay/A3TGCN-VAR-PEM-Implementation.git"
REPO_BRANCH = "main"
HORIZONS    = [1, 3, 6, 12]     # 5, 15, 30, 60 minutes
TOP_K       = 25                # sources kept per sensor (as E5)

## 1. Code and inputs

In [ ]:
import glob, json, shutil, subprocess, sys
import numpy as np

WORK = "/kaggle/working"
CODE = f"{WORK}/implementation"
if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
for needed in ("build_e8_graphs.py", "causal/var_pem_horizon.py"):
    assert os.path.exists(f"{CODE}/{needed}"), f"{needed} is not on GitHub yet: push the E8 files first."


def inputs(pattern):
    return sorted(p for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
                  if "/units/" not in p and "/implementation/" not in p)


def find_one(name):
    found = inputs(name)
    assert len(found) == 1, f"expected exactly one {name} under /kaggle/input, found {found}"
    return found[0]


# VAR files only (A_k, sensors, split). lingam_result.npz is deliberately NOT linked or opened.
VAR_DIR = f"{CODE}/outputs/var_train/lag3_alpha1e+06"
os.makedirs(VAR_DIR, exist_ok=True)
for name in ("coefficients.npy", "sensors.npy", "metadata.json"):
    os.symlink(find_one(name), f"{VAR_DIR}/{name}")


def prior_of(path):
    with np.load(path) as r:
        return str(r["prior"]) if "prior" in r.files else "uniform"


uniform = [p for p in inputs("pem_result*.npz") if prior_of(p) == "uniform"]
assert len(uniform) == 1, f"need exactly one uniform-prior pem_result*.npz (the E4 / E5 PEM run), found {uniform}"
PEM_RESULT = uniform[0]

E5_GRAPHS = f"{WORK}/e5_graphs_for_comparison"
os.makedirs(E5_GRAPHS, exist_ok=True)
for p in glob.glob("/kaggle/input/**/E5__h*.npz", recursive=True):
    shutil.copy(p, f"{E5_GRAPHS}/{os.path.basename(p)}")

print("PEM result :", PEM_RESULT)
print("VAR files  :", sorted(os.listdir(VAR_DIR)))
print("E5 graphs for the comparison:", len(os.listdir(E5_GRAPHS)), "files")

## 2. Build the E8 and E8V graphs

In [ ]:
OUT = f"{WORK}/e8_output"
os.makedirs(OUT, exist_ok=True)

cmd = [sys.executable, "-u", "build_e8_graphs.py", "--pem-result", PEM_RESULT,
       "--graph-dir", f"{OUT}/graphs", "--out-dir", f"{OUT}/e8", "--k", str(TOP_K),
       "--horizons", *map(str, HORIZONS)]
if os.listdir(E5_GRAPHS):
    cmd += ["--e5-graphs-dir", E5_GRAPHS]
print(" ".join(cmd))

log = open(f"{OUT}/build_e8.log", "w")
proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    log.write(line); log.flush()
    print(line.rstrip(), flush=True)
proc.wait()
log.close()
assert proc.returncode == 0, "the E8 build failed (see the log above); nothing is usable"

## 3. Diagnostics

In [ ]:
import pandas as pd
pd.set_option("display.width", 220, "display.max_columns", 30)
diag = json.load(open(f"{OUT}/e8/e8_diagnostics.json"))

print("LiNGAM / B0 access:", json.dumps(diag["lingam_access"], indent=1))
print("\nVAR:", json.dumps({k: diag["var"][k] for k in ("var_coefficient_shape", "impulse_response_shape")}))
print(pd.DataFrame(diag["var"]["per_horizon"]).T.round(5).to_string())
print("\nPEM probability matrix:", json.dumps(diag["pem"], indent=1))
for name in ("E8", "E8V"):
    print(f"\n{name} graphs (G[target, source], top-{TOP_K} per target):")
    print(pd.DataFrame(diag[name]).T.round(4).to_string())
print("\nChecks that stop the build:", json.dumps(diag["checks"], indent=1))
print("\nEdge overlap (Jaccard) E8 vs E8V:", diag["E8_vs_E8V_edge_overlap_jaccard"])
if "edge_overlap_with_stored_E5_graphs_(comparison_only)" in diag:
    print("Edge overlap with the stored E5 graphs (comparison only):",
          json.dumps(diag["edge_overlap_with_stored_E5_graphs_(comparison_only)"], indent=1))

## 4. In-degree of the graphs

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

cfg = json.load(open(f"{OUT}/e8/e8_config.json"))
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(3.2 * len(HORIZONS), 3.0), sharey=True)
for ax, h in zip(axes, HORIZONS):
    for name, color in (("E8", "#2a78d6"), ("E8V", "#e34948")):
        path = [f for f in cfg["graph_files"] if f.startswith(f"{name}__h{h}__")][0]
        with np.load(f"{OUT}/graphs/{path}") as a:
            indeg = (a["adjacency"] != 0).sum(axis=1)
        ax.hist(indeg, bins=np.arange(-0.5, TOP_K + 1.5, 1), alpha=0.55, color=color, label=name)
    ax.set(title=f"h = {h} ({5 * h} min)", xlabel="in-degree (sources per sensor)")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("sensors"); axes[0].legend(frameon=False)
plt.tight_layout(); plt.savefig(f"{OUT}/e8_in_degree.png", dpi=150); plt.show()

## 5. Collect the outputs and clean up

`e8_output/` holds the graph artifacts (`graphs/E8__h*.npz`, `graphs/E8V__h*.npz` and their `.json`), `e8/` (configuration,
diagnostics, VAR coefficients, PEM probabilities, impulse-response and effect matrices per horizon) and the build log.

In [ ]:
shutil.rmtree(CODE)
shutil.rmtree(E5_GRAPHS, ignore_errors=True)
for root, _, files in os.walk(OUT):
    for f in sorted(files):
        print(f"{os.path.getsize(os.path.join(root, f)) / 1e6:8.2f} MB  {os.path.relpath(os.path.join(root, f), OUT)}")
print("\nUpload graphs/E8__h*.npz and graphs/E8V__h*.npz to the artifacts dataset, then train with ablation-study.ipynb.")